# Rastgele Yapı Üreticiler

**Modüller:** `pytop.random_generators`, `pytop.random_relations`, `pytop.random_functions`  
**Konu:** Rastgele küme, topoloji, ilişki ve fonksiyon üretimi; yapısal garanti sağlayan üreticiler

---

Bu notebook altı bölümden oluşur:
1. **Konu** — Rastgele yapı neden önemli; yapısal garantili üretim
2. **Teoremler** — Warshall geçişli kapanımı, DAG-tabanlı kısmi sıralama
3. **Fonksiyon Analizi** — Pseudo-kod, karmaşıklık, avantaj/dezavantaj
4. **API** — `random_set`, `LazyTopology`, yapısal ilişki ve fonksiyon üreticiler
5. **Örnekler** — Dört senaryo: basit üretimden topolojik homeomorfizme
6. **Alıştırmalar** — Kodlama, teori ve karşılaştırma

---
## 1. Konu

### 1.1 Neden Rastgele Yapılar?

Matematikte birçok teorem "tüm X için" veya "bir X mevcuttur" biçiminde ifade edilir.  
Teoremi **sınamak** için somut örneklere ihtiyaç vardır. Elle oluşturmak yorucudur;
**rastgele üretim** yüzlerce test örneği üretir ve sezgimizi besler.

### 1.2 Yapısal Garanti

Sıradan `random` çağrıları geçerli matematik yapıları üretmeyebilir.  
Bu modüller her üretimde matematiksel koşulları **garantiler**:

| Üretici | Garanti |
|---------|--------|
| `random_reflexive_relation` | Köşegen her zaman dahil |
| `random_transitive_relation` | Warshall kapanımı uygulanır |
| `random_partial_order` | DAG yapısı → döngüsüz + geçişli |
| `random_total_order` | Rastgele permütasyon → doğrusal sıra |
| `random_equivalence_relation` | Rastgele bölüm → sınıflar |
| `random_injective_function` | Kodomain'den örnek alma → birebir |
| `random_bijection` | İki yönlü birebir eşleme |
| `random_continuous_function` | Taban kriteri ile süreklilik |
| `random_homeomorphism` | Sürekli + açık + bijektif |

### 1.3 `LazyTopology`

Büyük taşıyıcılar için tüm açık kümeler sayılamaz ($2^{2^n}$ adet olabilir).  
`LazyTopology` bir **altbaz** saklar ve açık küme sorgusunu talep üzerine yanıtlar:
$$U \text{ açık} \iff \forall x \in U,\ \exists B \in \mathcal{B}^*: x \in B \subseteq U$$
burada $\mathcal{B}^*$ altbazın kesişim kapanımıdır.

---
## 2. Teoremler

### Teorem 1 — Warshall Geçişli Kapanım Doğruluğu

**İfade:**  
Her sonlu ikili ilişki $R$ için Warshall algoritması, $R$'yi içeren ve geçişli olan en küçük ilişkiyi (geçişli kapanım $R^+$) üretir.

**İspat özeti:**  
Floyd-Warshall döngüsü, her $k$ için $k$-yolu kapsamlı olarak işler. Sabit nokta ulaşıldığında $mat[i][j] = 1$ tam olarak $i$'den $j$'ye yol varsa doğrudur. Dolayısıyla sonuç geçişlidir ve $R \subseteq R^+$. $\square$

---

### Teorem 2 — DAG'dan Kısmi Sıralama

**İfade:**  
Yönlü döngüsüz çizge (DAG) $G = (V, E)$ üzerinde $R = E^+$ (geçişli kapanım) yansımalı kapanımı alındığında kısmi sıralamadır.

**İspat:**
- *Yansımalı:* Köşegen eklenerek sağlanır.
- *Antisimetrik:* DAG döngüsüz olduğundan $u \to^* v$ ve $v \to^* u$ aynı anda geçerli olamaz; dolayısıyla $u = v$.
- *Geçişli:* Kapanım tanımı gereği. $\square$

---

### Teorem 3 — Taban Kriteri ile Süreklilik

**İfade:**  
$f: X \to Y$ süreklidir ancak ve ancak $Y$'nin her taban elemanının önresimi $X$'te açıktır.

**İspat:**  
$V$ açık $\Rightarrow$ $V = \bigcup_\alpha B_\alpha$ (taban elemanlarının birleşimi).  
$f^{-1}(V) = \bigcup_\alpha f^{-1}(B_\alpha)$ — her $f^{-1}(B_\alpha)$ açıksa birleşim de açıktır. $\square$

---
## 3. Fonksiyon Analizi

Aşağıda `pytop.random_generators` modülünün algoritmik üretici fonksiyonları incelenmektedir.

### `random_transitive_relation(carrier, density, seed)`

**Pseudo-kod (Warshall geçişli kapanım):**
1. `density` ve `seed`'e göre rastgele ilişki $R$ oluştur
2. $n \times n$ boolean matris `mat` tanımla; $(i,j) \in R$ ise `mat[i][j] = True`
3. Her `k ∈ {0..n-1}` için:
   - Her `(i,j)` çifti için: `mat[i][j] |= mat[i][k] and mat[k][j]`
4. `mat`'tan `frozenset` ilişkisi oluştur

**Karmaşıklık:**
- Zaman: O(n³) — Floyd-Warshall üç iç içe döngü
- Alan: O(n²) — boolean matris

**Avantajlar:**
- Geçişli kapanım garanti edilir; rastgelelik + yapısal doğruluk birleşir
- Küçük $n$ (≤20) için yeterince hızlı

**Dezavantajlar / Sınırlamalar:**
- Büyük $n$ için yavaş: `n=100` → 1M işlem
- Kapanım sonrası yoğunluk artar; istenen yoğunluğa yaklaşmak için girdi yoğunluğu azaltılmalı

---

### `random_partial_order(carrier, density, seed)`

**Pseudo-kod:**
1. Taşıyıcı üzerinde rastgele DAG oluştur (döngüsüz kenar seçimi — topological sort tabanlı)
2. Warshall ile geçişli kapanımı al
3. Köşegenı ekle (yansımalı kapat)
4. Sonucu döndür

**Karmaşıklık:**
- Zaman: O(n²) DAG oluşturma + O(n³) Warshall
- Alan: O(n²)

---

### `random_total_order(carrier, seed)` / `random_equivalence_relation(carrier, seed)`

**`random_total_order` pseudo-kod:**
1. `carrier`'ı rastgele karıştır (Fisher-Yates, O(n))
2. Permütasyona göre doğrusal sıra oluştur: $(π(i), π(j))$ ∀ $i < j$

**Karmaşıklık:** O(n²) — $n(n+1)/2$ çift

**`random_equivalence_relation`:** rastgele bölüm → her blok çifti denklik çifti: O(n²)

In [ ]:
from pytop.random_generators import (
    random_transitive_relation, random_partial_order,
    random_total_order, random_equivalence_relation,
)
from pytop.relations import is_transitive, is_partial_order, is_total_order, is_equivalence_relation

carrier = list(range(1, 6))

tr = random_transitive_relation(carrier, seed=42)
print(f"Geçişli ilişki:  {is_transitive(carrier, tr)}, |R|={len(tr)}")

po = random_partial_order(carrier, seed=42)
print(f"Kısmi sıralama: {is_partial_order(carrier, po)}, |R|={len(po)}")

to = random_total_order(carrier, seed=42)
print(f"Tam sıralama:   {is_total_order(carrier, to)}, |R|={len(to)} (n(n+1)/2={len(carrier)*(len(carrier)+1)//2})")

eq = random_equivalence_relation(carrier, seed=42)
print(f"Denklik:        {is_equivalence_relation(carrier, eq)}, |R|={len(eq)}")

---
## 4. API

In [ ]:
from pytop.random_generators import (
    random_set,
    random_topology,
    random_relation,
    random_function,
    LazyTopology,
    RandomGeneratorError,
    # Yapısal ilişki üreticiler
    random_reflexive_relation,
    random_symmetric_relation,
    random_transitive_relation,
    random_partial_order,
    random_total_order,
    random_equivalence_relation,
    # Yapısal fonksiyon üreticiler
    random_injective_function,
    random_surjective_function,
    random_bijection,
    random_continuous_function,
    random_homeomorphism,
)
from pytop.topology_builders import discrete_topology
from pytop.relations import is_partial_order, is_total_order, is_equivalence_relation
from pytop.maps import make_function
print("pytop yüklendi.")

### Temel Üreticiler

| Fonksiyon | Parametreler | Döndürür |
|-----------|-------------|----------|
| `random_set(max_size, ...)` | `size`, `element_type`, `seed` | `frozenset` |
| `random_topology(carrier, seed)` | `carrier`, `seed`, `max_subbasis_size` | `FiniteTopologicalSpace` veya `LazyTopology` |
| `random_relation(carrier, density, seed)` | `carrier`, `density` ∈ [0,1], `seed` | `set[tuple]` |
| `random_function(domain, codomain, seed)` | `domain`, `codomain`, `seed` | `dict` |

### Yapısal İlişki Üreticiler

| Fonksiyon | Garanti |
|-----------|--------|
| `random_reflexive_relation(carrier, density, seed)` | yansımalı |
| `random_symmetric_relation(carrier, density, seed)` | simetrik |
| `random_transitive_relation(carrier, density, seed)` | geçişli |
| `random_partial_order(carrier, density, seed)` | kısmi sıralama |
| `random_total_order(carrier, seed)` | tam sıralama |
| `random_equivalence_relation(carrier, seed)` | denklik ilişkisi |

### Yapısal Fonksiyon Üreticiler

| Fonksiyon | Garanti |
|-----------|--------|
| `random_injective_function(domain, codomain, seed)` | birebir |
| `random_surjective_function(domain, codomain, seed)` | örten |
| `random_bijection(carrier, seed)` | bijektif |
| `random_continuous_function(domain_sp, codomain_sp, seed)` | sürekli |
| `random_homeomorphism(domain_sp, codomain_sp, seed)` | homeomorfizm |

---
## 5. Örnekler

### Örnek 1 — Minimal: Rastgele Küme ve Topoloji

In [ ]:
# Rastgele kümeler
s1 = random_set(max_size=5, seed=42)
s2 = random_set(size=4, element_type='char', seed=7)
s3 = random_set(size=3, element_type='int', random_order=False)  # ilk 3 eleman

print(f"Rastgele int kümesi (max=5):   {s1}")
print(f"Rastgele char kümesi (boyut=4): {s2}")
print(f"Sıralı pool'dan 3 eleman:       {s3}")

# Rastgele topoloji (küçük taşıyıcı → FiniteTopologicalSpace)
carrier = [1, 2, 3]
tau = random_topology(carrier, seed=0)
print(f"\nRastgele topoloji türü: {type(tau).__name__}")
print(f"Açık küme sayısı: {len(tau.topology)}")

### Örnek 2 — Orta Düzey: Yapısal İlişki Üreticiler

In [ ]:
carrier = list(range(1, 6))   # {1,2,3,4,5}

po = random_partial_order(carrier, seed=10)
to = random_total_order(carrier, seed=10)
eq = random_equivalence_relation(carrier, seed=10)

print(f"Kısmi sıralama doğrulama:   {is_partial_order(carrier, po)}")
print(f"Tam sıralama doğrulama:     {is_total_order(carrier, to)}")
print(f"Denklik ilişkisi doğrulama: {is_equivalence_relation(carrier, eq)}")

# Farklı tohumlarda farklı yapılar
orders = [random_total_order(carrier, seed=s) for s in range(5)]
print(f"\n5 farklı tam sıralamada hepsi geçerli: {all(is_total_order(carrier, o) for o in orders)}")

### Örnek 3 — Gelişmiş: LazyTopology ve Açık Küme Sorgusu

In [ ]:
# Büyük taşıyıcı → LazyTopology
big_carrier = list(range(10))
lazy = random_topology(big_carrier, seed=42, max_subbasis_size=4)
print(f"Topoloji türü: {type(lazy).__name__}")
print(f"UID: {lazy.uid}")
print(f"Altbaz boyutu: {len(lazy.subbasis)}")

# Açık küme sorgusu
empty_open = lazy.is_open([])
full_open  = lazy.is_open(big_carrier)
print(f"\n∅ açık mı? {empty_open}")
print(f"X açık mı? {full_open}")

# Rastgele açık kümeler
opens = lazy.sample_open_sets(3, seed=0)
for i, U in enumerate(opens):
    print(f"Açık küme {i+1}: {sorted(U)} — doğrulama: {lazy.is_open(U)}")

# UID ile yeniden oluşturma
lazy2 = LazyTopology.from_uid(big_carrier, lazy.uid)
print(f"\nUID'den yeniden oluşturma tutarlı mı? {lazy.uid == lazy2.uid}")

### Örnek 4 — Gerçekçi Senaryo: Topolojik Fonksiyon Üreticiler

In [ ]:
# Ayrık topoloji → her fonksiyon sürekli
disc3  = discrete_topology(1, 2, 3)
disc3b = discrete_topology(1, 2, 3)

# Rastgele sürekli fonksiyon (ayrık → ayrık: her fonksiyon süreklidir)
try:
    f_dict = random_continuous_function(disc3, disc3b, seed=99)
    print(f"Sürekli fonksiyon: {f_dict}")
except RandomGeneratorError as e:
    print(f"Üretici bulunamadı: {e}")

# Rastgele bijeksiyon — codomain açıkça belirtilmeli
bij = random_bijection([1, 2, 3], [1, 2, 3], seed=42)
print(f"Bijeksiyon: {bij}")

# make_function ile FiniteMap'e dönüştürme
fm = make_function([1, 2, 3], [1, 2, 3], bij)
print(f"FiniteMap oluşturuldu: {fm}")

# Enjektif ve örten fonksiyon üretimi
inj = random_injective_function([1, 2], [1, 2, 3, 4], seed=5)
sur = random_surjective_function([1, 2, 3, 4], [1, 2], seed=5)
print(f"\nEnjektif (domain=[1,2], codomain=[1,2,3,4]): {inj}")
print(f"Örten   (domain=[1,2,3,4], codomain=[1,2]):   {sur}")

---
## 6. Alıştırmalar

### Alıştırma 1 — Kodlama Görevi

$\{a, b, c, d, e\}$ taşıyıcısı üzerinde:
1. 10 farklı `random_partial_order` üret (seed=0…9)
2. Her birinin geçerli kısmi sıralama olduğunu doğrula
3. Ortalama ilişki yoğunluğunu (çift sayısı / $n^2$) hesapla

In [ ]:
c5 = list('abcde')
n = len(c5)
orders = [random_partial_order(c5, seed=s) for s in range(10)]
all_valid = all(is_partial_order(c5, o) for o in orders)
avg_density = sum(len(o) for o in orders) / (10 * n * n)
print(f"Hepsi geçerli kısmi sıralama: {all_valid}")
print(f"Ortalama yoğunluk: {avg_density:.3f}")

### Alıştırma 2 — Teori Sorusu

a) `random_transitive_relation` ile `random_partial_order` arasındaki fark nedir?  
b) `LazyTopology` neden büyük taşıyıcılar için tercih edilir?  
c) Ayrık topoloji üzerinde her fonksiyon sürekli midir? Neden?

_Yanıtlarınızı buraya yazın:_

a) ...

b) ...

c) ...

### Alıştırma 3 — Karşılaştırma

`random_relation(density=0.3)` ile `random_reflexive_relation(density=0.3)` üret.  
100 deney sonucunda:
- Birincinin kaçı kendiliğinden yansımalıdır?
- İkincisinin kaçı yansımalıdır? (Her biri yansımalı olmalı)

In [ ]:
from pytop.relations import is_reflexive

c4 = [1, 2, 3, 4]
n_trials = 100

plain_refl  = sum(is_reflexive(c4, random_relation(c4, seed=s, density=0.3)) for s in range(n_trials))
struct_refl = sum(is_reflexive(c4, random_reflexive_relation(c4, density=0.3, seed=s)) for s in range(n_trials))

print(f"Sıradan (density=0.3): {plain_refl}/{n_trials} yansımalı")
print(f"Yapısal:               {struct_refl}/{n_trials} yansımalı")

### Alıştırma 4 — Karmaşıklık Ölçümü

`random_transitive_relation`'ın O(n³) Warshall maliyetini deneysel olarak doğrulayın:
1. `n ∈ {5, 10, 20, 40}` için `random_transitive_relation(list(range(n)), seed=0)` çağrısını `timeit` ile ölçün
2. Her `n` için `t(n)` değerini yazdırın
3. `t(2n) / t(n)` oranını hesaplayın — O(n³) için yaklaşık 8 beklenir
4. `random_partial_order` ile `random_equivalence_relation` maliyetlerini karşılaştırın (`n=20`)

In [ ]:
import timeit

ns = [5, 10, 20, 40]
times_tr = []
for n in ns:
    carrier_n = list(range(n))
    t = timeit.timeit(lambda c=carrier_n: random_transitive_relation(c, seed=0), number=20) / 20
    times_tr.append(t)
    print(f"n={n:2d}  t={t*1000:.3f} ms")

print("\nt(2n)/t(n) oranları (O(n³) → ~8 beklenir):")
for i in range(1, len(ns)):
    ratio = times_tr[i] / times_tr[i-1]
    print(f"  n={ns[i-1]}→{ns[i]}: {ratio:.2f}x")

# n=20 karşılaştırması
c20 = list(range(20))
t_po  = timeit.timeit(lambda: random_partial_order(c20, seed=0), number=20) / 20
t_eq  = timeit.timeit(lambda: random_equivalence_relation(c20, seed=0), number=20) / 20
print(f"\nn=20 karşılaştırma:")
print(f"  random_partial_order:       {t_po*1000:.3f} ms")
print(f"  random_equivalence_relation:{t_eq*1000:.3f} ms")

---
## Özet

| Kavram | Açıklama |
|--------|----------|
| `random_set` | `frozenset`; boyut/tür/tohum kontrolü |
| `random_topology` | Küçük n → `FiniteTopologicalSpace`; büyük n → `LazyTopology` (altbaz saklar) |
| `random_transitive_relation` | Warshall O(n³); geçişli kapanım garantisi |
| `random_partial_order` | DAG O(n²) + Warshall O(n³); yansımalı+antisimetrik+geçişli |
| `random_total_order` | Fisher-Yates permütasyon O(n) → n(n+1)/2 çift |
| `random_equivalence_relation` | Rastgele bölüm → sınıflar → O(n²) |
| `random_injective_function` | Codomain'den örneksiz seçim; birebir garanti |
| `random_bijection` | İki yönlü; `random_injective_function` özel hali |
| `random_continuous_function` | Taban kriteri; ayrık → her fonksiyon sürekli |
| `random_homeomorphism` | Sürekli + açık + bijektif |
| `LazyTopology` | Altbaz sakla; açık sorgusu O(|subbasis|) |

**Sonraki konu:** `pytop.homotopy` — yollar, homotopi denkliği, temel grup